## 1. Загружаем данные

> Download data from Don’tGetKicked competition.

Задача: предсказать, окажется ли купленный на аукционе подержанный автомобиль "лимоном" (IsBadBuy = 1), то есть машиной со скрытыми проблемами, которую дилеру невыгодно перепродавать.

Файл `training.csv` это обучающая выборка из соревнования Don't Get Kicked (https://www.kaggle.com/c/DontGetKicked).

Целевая переменная `IsBadBuy` (1 = машина оказалась "лимоном", 0 = нормальная покупка).

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

from sklearn.metrics import accuracy_score

pd.set_option('display.max_columns', 50)

In [2]:
df = pd.read_csv('./data/training.csv')
print("Размер датасета:", df.shape)
df.head()

Размер датасета: (72983, 34)


,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,WheelTypeID,WheelType,VehOdo,Nationality,Size,TopThreeAmericanName,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,MMRAcquisitonRetailCleanPrice,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,RED,AUTO,1.0,Alloy,89046,OTHER ASIAN,MEDIUM,OTHER,8155.0,9829.0,11636.0,13600.0,7451.0,8552.0,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,WHITE,AUTO,1.0,Alloy,93593,AMERICAN,LARGE TRUCK,CHRYSLER,6854.0,8383.0,10897.0,12572.0,7456.0,9222.0,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,MAROON,AUTO,2.0,Covers,73807,AMERICAN,MEDIUM,CHRYSLER,3202.0,4760.0,6943.0,8457.0,4035.0,5557.0,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,SILVER,AUTO,1.0,Alloy,65617,AMERICAN,COMPACT,CHRYSLER,1893.0,2675.0,4658.0,5690.0,1844.0,2646.0,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,SILVER,MANUAL,2.0,Covers,69367,AMERICAN,COMPACT,FORD,3913.0,5054.0,7723.0,8707.0,3247.0,4384.0,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020


Колонки

In [3]:
list(df.columns)

['RefId',
 'IsBadBuy',
 'PurchDate',
 'Auction',
 'VehYear',
 'VehicleAge',
 'Make',
 'Model',
 'Trim',
 'SubModel',
 'Color',
 'Transmission',
 'WheelTypeID',
 'WheelType',
 'VehOdo',
 'Nationality',
 'Size',
 'TopThreeAmericanName',
 'MMRAcquisitionAuctionAveragePrice',
 'MMRAcquisitionAuctionCleanPrice',
 'MMRAcquisitionRetailAveragePrice',
 'MMRAcquisitonRetailCleanPrice',
 'MMRCurrentAuctionAveragePrice',
 'MMRCurrentAuctionCleanPrice',
 'MMRCurrentRetailAveragePrice',
 'MMRCurrentRetailCleanPrice',
 'PRIMEUNIT',
 'AUCGUART',
 'BYRNO',
 'VNZIP1',
 'VNST',
 'VehBCost',
 'IsOnlineSale',
 'WarrantyCost']

Доля плохих покупок (IsBadBuy=1)

In [4]:
df['IsBadBuy'].mean().round(4)

np.float64(0.123)

Пропуски в данных

In [5]:
df.isna().sum()[df.isna().sum() > 0]

Trim                                  2360
SubModel                                 8
Color                                    8
Transmission                             9
WheelTypeID                           3169
WheelType                             3174
Nationality                              5
Size                                     5
TopThreeAmericanName                     5
MMRAcquisitionAuctionAveragePrice       18
MMRAcquisitionAuctionCleanPrice         18
MMRAcquisitionRetailAveragePrice        18
MMRAcquisitonRetailCleanPrice           18
MMRCurrentAuctionAveragePrice          315
MMRCurrentAuctionCleanPrice            315
MMRCurrentRetailAveragePrice           315
MMRCurrentRetailCleanPrice             315
PRIMEUNIT                            69564
AUCGUART                             69564
dtype: int64

Почти во всех столбцах пропусков достаточно мало, но `PRIMEUNIT` и `AUCGUART` пустые почти у всех строк (69564 из 72983). С ними разберемся отдельно на этапе препроцессинга.

## 2. Делим данные на train / valid / test по дате покупки

> Design the train/validation/test split. Use the "PurchDate" field for the split, test must be later than validation, same for validation and train: train.PurchDate < valid.PurchDate < test.PurchDate. Use the first 1/3 of dates for the train, the last 1/3 of dates for the test, and the middle 1/3 for the validation set. Don’t use the test dataset until the end!

По условию: `train.PurchDate < valid.PurchDate < test.PurchDate`.

Берем весь диапазон дат покупки (`PurchDate`) и делим его на 3 равные по времени части:
- первая треть периода `train`
- средняя треть `valid`
- последняя треть `test`

Модель обучается на прошлом, а проверяется на будущем. Тестовую выборку не трогаем до пункта 10.

In [6]:
df['PurchDate'] = pd.to_datetime(df['PurchDate'])

date_min = df['PurchDate'].min()
date_max = df['PurchDate'].max()
span = date_max - date_min

# границы третей периода
border_1 = date_min + span / 3
border_2 = date_min + 2 * span / 3

train = df[df['PurchDate'] < border_1].copy()
valid = df[(df['PurchDate'] >= border_1) & (df['PurchDate'] < border_2)].copy()
test  = df[df['PurchDate'] >= border_2].copy()

In [7]:
print("Границы дат: ", date_min.date(), "|", border_1.date(), "|", border_2.date(), "|", date_max.date())

Границы дат:  2009-01-05 | 2009-09-03 | 2010-05-02 | 2010-12-30


In [8]:
print("train:", train.shape, "период:", train['PurchDate'].min().date(), "-", train['PurchDate'].max().date())
print("valid:", valid.shape, "период:", valid['PurchDate'].min().date(), "-", valid['PurchDate'].max().date())
print("test: ", test.shape, "период:", test['PurchDate'].min().date(), "-", test['PurchDate'].max().date())

train: (23483, 34) период: 2009-01-05 - 2009-09-03
valid: (23680, 34) период: 2009-09-04 - 2010-04-30
test:  (25820, 34) период: 2010-05-03 - 2010-12-30


In [9]:
assert train['PurchDate'].max() <= valid['PurchDate'].min()
assert valid['PurchDate'].max() <= test['PurchDate'].min()

print("Доля IsBadBuy: train=%.4f valid=%.4f test=%.4f" % (train['IsBadBuy'].mean(), valid['IsBadBuy'].mean(), test['IsBadBuy'].mean()))

Доля IsBadBuy: train=0.1136 valid=0.1327 test=0.1226


Доля "плохих" машин во всех трех частях примерно одинаковая (11-13%), значит выборки сбалансированы похоже.

## 3. Препроцессинг признаков

> Use LabelEncoder or OneHotEncoder from sklearn to preprocess categorical variables. Be careful with data leakage (fit Encoder to training and apply to validation & test). Consider another coding approach if you encounter new categorical values in validation & test (not seen in training): https://contrib.scikit-learn.org/category_encoders/count.html

- убираем служебные и "дырявые" колонки (`RefId` просто номер записи, `PRIMEUNIT` и `AUCGUART` почти полностью пустые)
- категориальные признаки кодируем `LabelEncoder`, а для числовых заполняем пропуски медианой, посчитанной только по train, чтобы не было утечки данных
- энкодер (`LabelEncoder`) обучаем (`fit`) только на train, а затем просто применяем (`transform`) к valid и test, так модель "не видела" будущее заранее
- если в valid/test встретится значение категории, которого не было в train (например, новая модель машины), `LabelEncoder` сломается. обрабатываем вручную: таким новым значениям присваиваем код `-1` ("неизвестная категория"). альтернативный подход `CountEncoder` из библиотеки `category_encoders` (https://contrib.scikit-learn.org/category_encoders/count.html), который кодирует категорию просто частотой ее встречаемости в train, тогда новая категория получает значение по умолчанию, а не падает с ошибкой.

In [10]:
target = 'IsBadBuy'

# колонки которые не используем как признаки
drop_cols = ['RefId', 'PurchDate', 'IsBadBuy', 'PRIMEUNIT', 'AUCGUART']

# категориальные признаки
cat_cols = ['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'VNST']

# все остальное - числовые признаки
num_cols = [c for c in df.columns if c not in drop_cols + cat_cols]

Числовые признаки

In [11]:
print(num_cols)

['VehYear', 'VehicleAge', 'WheelTypeID', 'VehOdo', 'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'BYRNO', 'VNZIP1', 'VehBCost', 'IsOnlineSale', 'WarrantyCost']


Категориальные признаки

In [12]:
print(cat_cols)

['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'VNST']


Заполняем пропуски

In [13]:
for c in cat_cols:
    for part in (train, valid, test):
        part[c] = part[c].fillna('MISSING').astype(str)

for c in num_cols:
    median_value = train[c].median() # медиана только по трейну
    for part in (train, valid, test):
        part[c] = part[c].fillna(median_value)

Пропуски после обработки в train

In [14]:
print(train[num_cols + cat_cols].isna().sum().sum())

0


In [15]:
encoders = {}
UNKNOWN_CODE = -1

for c in cat_cols:
    le = LabelEncoder()
    le.fit(train[c]) # фит только на трейне
    encoders[c] = (le, set(le.classes_))
    train[c + '_enc'] = le.transform(train[c])

for c in cat_cols:
    le, known_classes = encoders[c]
    for part in (valid, test):
        part[c + '_enc'] = part[c].apply(
            lambda value: le.transform([value])[0] if value in known_classes else UNKNOWN_CODE
        )

feature_cols = num_cols + [c + '_enc' for c in cat_cols]

Итоговое число признаков

In [16]:
print(len(feature_cols))

29


сколько новых (не встречавшихся в train) категорий нашлось в valid и test

In [17]:
for c in cat_cols:
    n_unseen_valid = (valid[c + '_enc'] == UNKNOWN_CODE).sum()
    n_unseen_test = (test[c + '_enc'] == UNKNOWN_CODE).sum()
    if n_unseen_valid or n_unseen_test:
        print(f"{c}: новых категорий в valid={n_unseen_valid}, в test={n_unseen_test}")

Make: новых категорий в valid=8, в test=30
Model: новых категорий в valid=4088, в test=5975
Trim: новых категорий в valid=15, в test=25
SubModel: новых категорий в valid=236, в test=376
Transmission: новых категорий в valid=1, в test=0
Nationality: новых категорий в valid=5, в test=0
Size: новых категорий в valid=5, в test=0
TopThreeAmericanName: новых категорий в valid=5, в test=0
VNST: новых категорий в valid=44, в test=723


В некоторых признаках и правда встречаются новые значения, которых не было в трейне.

Итого, категориальные признаки закодированы с помощью `LabelEncoder`, обучение происходило только на трейне, для новых категорий в valid/test предусмотрена обработка (код `-1`)

## 4. Обучаем LogisticRegression, GaussianNB, KNN

> Train LogisticRegression, GaussianNB, KNN from sklearn on the training dataset and check the quality of your algorithms on the validation dataset. The dependent variable (IsBadBuy) is binary. Don't forget to normalize your datasets before training your models.
>
> You must get at least 0.15 Gini score (the best of all three). Which algorithm performs better? And why?

Перед обучением нормализуем признаки со `StandardScaler`, тк это важно для KNN (основан на расстояниях между точками) и логистической регрессии (быстрее сходится).

скейлер, как и энкодер, обучаем только на трейне.

Целевая метрика **Gini = 2 * ROC AUC - 1**. Нужно получить Gini >= 0.15 хотя бы для одной модели.

In [18]:
X_train = train[feature_cols].values.astype(float)
X_valid = valid[feature_cols].values.astype(float)
X_test  = test[feature_cols].values.astype(float)

y_train = train[target].values
y_valid = valid[target].values
y_test  = test[target].values

scaler = StandardScaler()
scaler.fit(X_train) # фит скейлера только на трейне

X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s  = scaler.transform(X_test)

In [19]:
def gini_score(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1

Получившиеся данные

In [20]:
print(X_train_s.shape, X_valid_s.shape, X_test_s.shape)

(23483, 29) (23680, 29) (25820, 29)


In [21]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'GaussianNB': GaussianNB(),
    'KNN (k=25)': KNeighborsClassifier(n_neighbors=25),
}

baseline_gini = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    proba_valid = model.predict_proba(X_valid_s)[:, 1]
    g = gini_score(y_valid, proba_valid)
    baseline_gini[name] = g
    print(f"{name:22s}  Gini (valid) = {g:.4f}")

LogisticRegression      Gini (valid) = 0.4727
GaussianNB              Gini (valid) = 0.3475
KNN (k=25)              Gini (valid) = 0.3990


Все три модели обучены на train и проверены на valid, у всех Gini >= 0.15

Лучше всего себя показала логистическая регрессия, потому что:
- признаки в основном либо числовые монотонно-связанные с целью (цены MMR, пробег, возраст машины), либо закодированные категории, поэтому линейная модель с хорошей регуляризацией работает хорошо и не переобучается.
- KNN страдает от проклятия размерности из-за того что у нас довольно много признаков (~30), и расстояния между точками в таком пространстве становятся менее информативными; плюс KNN чувствителен к неинформативным/шумным признакам (а `LabelEncoder`-категории это по сути произвольное число, не несущее "расстояния" в содержательном смысле).
- GaussianNB предполагает, что признаки независимы и распределены нормально внутри класса, но здесь это нарушаются (признаки коррелируют друг с другом, например, цены MMR между собой сильно связаны, закодированные категории не нормальны), поэтому качество ниже, чем у логистической регрессии.

## 5. Реализуем расчет Gini самостоятельно

> Implement Gini score calculation. You can use the 2*ROC AUC - 1 approach, so you need to implement the ROC AUC calculation. Check if your metric is approximately equal to `abs(2*sklearn.metrics.roc_auc_score - 1)`.

Gini = 2 * ROC AUC - 1, поэтому реализуем расчет ROC AUC руками по определению:
- сортируем предсказания по убыванию скора.
- идем по отсортированному списку и на каждом шаге считаем True Positive Rate (TPR) и False Positive Rate (FPR), долю верно и неверно найденных положительных примеров.
- ROC AUC это площадь под кривой (TPR по оси Y, FPR по оси X), считаем ее методом трапеций.

In [22]:
def my_roc_auc(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    # сортируем объекты по убыванию предсказанного скора
    order = np.argsort(-y_score)
    y_true_sorted = y_true[order]

    n_pos = y_true_sorted.sum()
    n_neg = len(y_true_sorted) - n_pos

    # накопленное число верно и неверно предсказанных положительных
    tp_cumsum = np.cumsum(y_true_sorted)
    fp_cumsum = np.cumsum(1 - y_true_sorted)

    tpr = tp_cumsum / n_pos
    fpr = fp_cumsum / n_neg

    # добавляем точку (0, 0) в начало кривой
    tpr = np.concatenate(([0.0], tpr))
    fpr = np.concatenate(([0.0], fpr))

    # площадь под ROC-кривой
    auc = np.trapezoid(tpr, fpr)
    return auc

def my_gini(y_true, y_score):
    return 2 * my_roc_auc(y_true, y_score) - 1

Проверяем, что my_roc_auc() и my_gini() совпадают с sklearn

In [23]:
for name, model in models.items():
    proba_valid = model.predict_proba(X_valid_s)[:, 1]

    sk_auc = roc_auc_score(y_valid, proba_valid)
    my_auc = my_roc_auc(y_valid, proba_valid)

    sk_gini = abs(2 * sk_auc - 1)
    my_g = my_gini(y_valid, proba_valid)

    print(f"{name:22s}  sklearn AUC={sk_auc:.6f}  my AUC={my_auc:.6f}  |  "
          f"sklearn Gini={sk_gini:.6f}  my Gini={my_g:.6f}")

LogisticRegression      sklearn AUC=0.736331  my AUC=0.736331  |  sklearn Gini=0.472661  my Gini=0.472661
GaussianNB              sklearn AUC=0.673727  my AUC=0.673727  |  sklearn Gini=0.347454  my Gini=0.347454
KNN (k=25)              sklearn AUC=0.699481  my AUC=0.698982  |  sklearn Gini=0.398961  my Gini=0.397964


Реализация ROC AUC и Gini совпадает с sklearn.

## 6. Свои реализации LogisticRegression, KNN и NaiveBayes

> Implement your own versions of LogisticRegression, KNN and NaiveBayes classifiers. For LogisticRegression compute gradients with respect to the loss and use stochastic gradient descent. Can you reproduce the results from step 4?
>
> Guidance for this task: Your model must be represented by class with methods fit, predict (predict_proba with 0.5 threshold), predict_proba. For LR moder, compute the loss gradient with respect to parameters w and parameter b in the fit function. Use a simple SGD approach to estimate optimal values of parameters.

Пишем три класса с методами `fit`, `predict`, `predict_proba` как в sklearn.

### 6.1 Логистическая регрессия (через SGD)

Модель: `p = sigmoid(X @ w + b)`.

Функция потерь логистическая (log loss / binary cross-entropy):

`L = -mean( y*log(p) + (1-y)*log(1-p) )`

Градиенты по параметрам (выводятся аналитически, получаются очень простыми):

`dL/dw = X.T @ (p - y) / n`,  `dL/db = mean(p - y)`

Чтобы было быстрее, обучаем не по всему датасету сразу, а мини-батчами (стохастический градиентный спуск, SGD).

In [24]:
class MyLogisticRegression:
    def __init__(self, lr=0.3, n_epochs=30, batch_size=128, l2=0.0, random_state=42):
        self.lr = lr # скорость обучения
        self.n_epochs = n_epochs # число проходов по данным
        self.batch_size = batch_size
        self.l2 = l2 # сила L2-регуляризации
        self.random_state = random_state

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500) # чтобы не было переполнения exp()
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        for epoch in range(self.n_epochs):
            order = rng.permutation(n_samples) # перемешиваем данные каждую эпоху
            for start in range(0, n_samples, self.batch_size):
                batch_idx = order[start:start + self.batch_size]
                X_batch, y_batch = X[batch_idx], y[batch_idx]

                p = self._sigmoid(X_batch @ self.w + self.b)

                # градиент функции потерь по w и по b
                grad_w = X_batch.T @ (p - y_batch) / len(y_batch) + self.l2 * self.w
                grad_b = np.mean(p - y_batch)

                # шаг градиентного спуска
                self.w -= self.lr * grad_w
                self.b -= self.lr * grad_b
        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

### 6.2 KNN (метод k ближайших соседей)

`predict_proba` для объекта = доля класса 1 среди k ближайших соседей в train.
Расстояние считаем евклидово, но не в цикле по одному объекту, а векторизованно (через формулу
`||a-b||^2 = ||a||^2 + ||b||^2 - 2*a·b`), чтобы было быстрее.

In [25]:
class MyKNN:
    def __init__(self, k=15):
        self.k = k

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self

    def predict_proba(self, X, batch_size=200):
        n = len(X)
        proba = np.zeros(n)
        train_sq = np.sum(self.X_train ** 2, axis=1)

        for start in range(0, n, batch_size):
            X_batch = X[start:start + batch_size]
            batch_sq = np.sum(X_batch ** 2, axis=1)[:, None]
            # квадраты расстояний до всех точек train, для всего батча сразу
            dist_sq = batch_sq + train_sq[None, :] - 2 * X_batch @ self.X_train.T
            nearest_idx = np.argsort(dist_sq, axis=1)[:, :self.k]
            proba[start:start + batch_size] = self.y_train[nearest_idx].mean(axis=1)
        return proba

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

### 6.3 Наивный байесовский классификатор (Gaussian Naive Bayes)

Предполагаем, что внутри каждого класса каждый признак распределен нормально (гауссиана) и признаки независимы друг от друга. Тогда по формуле Байеса:

`P(class=c | x) ~ P(class=c) * произведение по признакам P(x_i | class=c)`

Работаем в логарифмах чтобы не перемножать много маленьких чисел и не терять точность.

In [26]:
class MyGaussianNB:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.priors = {} # P(class=c)
        self.mean = {} # среднее признаков внутри класса
        self.var = {} # дисперсия признаков внутри класса

        for c in self.classes:
            X_c = X[y == c]
            self.priors[c] = len(X_c) / len(X)
            self.mean[c] = X_c.mean(axis=0)
            self.var[c] = X_c.var(axis=0) + 1e-9 # +eps, чтобы не делить на 0

        return self

    def _log_likelihood(self, X, c):
        mean, var = self.mean[c], self.var[c]
        # логарифм плотности нормального распределения просуммированный по всем признакам
        return -0.5 * np.sum(np.log(2 * np.pi * var)) - 0.5 * np.sum(((X - mean) ** 2) / var, axis=1)

    def predict_proba(self, X):
        log_scores = []
        for c in self.classes:
            log_scores.append(np.log(self.priors[c]) + self._log_likelihood(X, c))
        log_scores = np.array(log_scores).T
        log_scores -= log_scores.max(axis=1, keepdims=True) # для численной стабильности
        probs = np.exp(log_scores)
        probs /= probs.sum(axis=1, keepdims=True)
        return probs[:, 1] # вероятность класса 1

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

Проверяем, получилось ли воспроизвести результаты пункта 4.

In [27]:
my_lr = MyLogisticRegression(lr=0.3, n_epochs=30, batch_size=128).fit(X_train_s, y_train)
my_nb = MyGaussianNB().fit(X_train_s, y_train)

my_lr_gini = my_gini(y_valid, my_lr.predict_proba(X_valid_s))
my_nb_gini = my_gini(y_valid, my_nb.predict_proba(X_valid_s))

print("Своя LogisticRegression: Gini (valid) = %.4f (sklearn: %.4f)" % (my_lr_gini, baseline_gini['LogisticRegression']))
print("Своя GaussianNB: Gini (valid) = %.4f (sklearn: %.4f)" % (my_nb_gini, baseline_gini['GaussianNB']))

Своя LogisticRegression: Gini (valid) = 0.4687 (sklearn: 0.4727)
Своя GaussianNB: Gini (valid) = 0.3475 (sklearn: 0.3475)


KNN на подвыборке (полный перебор расстояний вручную на 20к+ строк будет медленным)

In [28]:
rng = np.random.RandomState(0)
train_sub_idx = rng.choice(len(X_train_s), 3000, replace=False)
valid_sub_idx = rng.choice(len(X_valid_s), 2000, replace=False)

X_train_sub, y_train_sub = X_train_s[train_sub_idx], y_train[train_sub_idx]
X_valid_sub, y_valid_sub = X_valid_s[valid_sub_idx], y_valid[valid_sub_idx]

my_knn = MyKNN(k=25).fit(X_train_sub, y_train_sub)
sk_knn_sub = KNeighborsClassifier(n_neighbors=25).fit(X_train_sub, y_train_sub)

my_knn_gini = my_gini(y_valid_sub, my_knn.predict_proba(X_valid_sub))
sk_knn_gini_sub = gini_score(y_valid_sub, sk_knn_sub.predict_proba(X_valid_sub)[:, 1])

print("Своя KNN: Gini (valid, подвыборка) = %.4f" % my_knn_gini)
print("sklearn: KNN Gini (valid, подвыборка) = %.4f" % sk_knn_gini_sub)

Своя KNN: Gini (valid, подвыборка) = 0.4328
sklearn: KNN Gini (valid, подвыборка) = 0.4293


Результаты по Gini очень близки к sklearn-версиям (небольшие расхождения у логрегрессии из-за того, что sklearn по умолчанию использует другой оптимизатор (`lbfgs`) и L2-регуляризацию, а у нас простой SGD без регуляризации по умолчанию).

## 7. Добавляем нелинейные признаки

> Try to create non-linear features, for example:
> 
> fractions: feature1/feature2
> 
> groupby features: df[‘categorical_feature’].map(df.groupby(‘categorical_feature’)[‘continious_feature’].mean())
> 
> Add new features to your pipeline, repeat step 4. Did you manage to increase your Gini score (you should!)?

Просто числовые колонки не всегда лучший вход для линейной модели, тк иногда важнее не сама цена, а отношение цен или разница между ними (например, насколько текущая аукционная цена "просела" по сравнению с ценой покупки может говорить о скрытых проблемах машины).

Добавляем:
- Отношения (fractions): пробег на "год жизни" машины (`VehOdo / VehicleAge`), отношение текущей/розничной цены к цене на момент покупки, эдакая скорость обесценивания.
- Групповые признаки (groupby): средняя цена покупки для конкретной марки (`Make`), сравнение цены конкретного авто со средней ценой по его марке.

Эти статистики (медианы, средние по группам) считаем только по трейну, чтобы избежать утечки.

In [29]:
def add_engineered_features(part, make_mean_cost, global_mean_cost):
    part = part.copy()
    # пробег на год жизни машины
    part['Odo_per_Age'] = part['VehOdo'] / (part['VehicleAge'] + 1)

    # во сколько раз текущая цена отличается от цены на момент закупки (скорость обесценивания)
    part['Ratio_auction_price'] = part['MMRCurrentAuctionAveragePrice'] / (part['MMRAcquisitionAuctionAveragePrice'] + 1)
    part['Ratio_retail_price']  = part['MMRCurrentRetailAveragePrice']  / (part['MMRAcquisitionRetailAveragePrice']  + 1)

    # во сколько раз цена покупки отличается от справочной аукционной цены (переплатили или недоплатили)
    part['Cost_vs_MMR'] = part['VehBCost'] / (part['MMRAcquisitionAuctionAveragePrice'] + 1)

    # групповой признак, средняя цена покупки для марки машины (Make)
    part['Make_mean_cost'] = part['Make'].map(make_mean_cost).fillna(global_mean_cost)

    return part

Считаем групповую статистику только по трейну

In [30]:
make_mean_cost = train.groupby('Make')['VehBCost'].mean()
global_mean_cost = train['VehBCost'].mean()

train_fe = add_engineered_features(train, make_mean_cost, global_mean_cost)
valid_fe = add_engineered_features(valid, make_mean_cost, global_mean_cost)
test_fe  = add_engineered_features(test,  make_mean_cost, global_mean_cost)

new_features = ['Odo_per_Age', 'Ratio_auction_price', 'Ratio_retail_price', 'Cost_vs_MMR', 'Make_mean_cost']

# чистим возможные inf/NaN
for c in new_features:
    median_value = train_fe[c].replace([np.inf, -np.inf], np.nan).median()
    for part in (train_fe, valid_fe, test_fe):
        part[c] = part[c].replace([np.inf, -np.inf], np.nan).fillna(median_value)

train_fe[new_features].describe()

,Odo_per_Age,Ratio_auction_price,Ratio_retail_price,Cost_vs_MMR,Make_mean_cost
count,23483.000000,23483.000000,23483.000000,23483.000000,23483.000000
mean,14628.390713,5.057767,5.847605,94.596392,6346.200851
std,5046.713169,173.389524,206.198581,793.994784,612.748514
min,766.857143,0.000000,0.000000,0.000415,3560.000000
25%,11054.916667,0.981887,0.993164,0.961635,6070.627986
50%,13764.000000,0.999854,0.999886,1.112910,6473.764035
75%,17236.250000,1.046242,1.053609,1.337822,6503.225016
max,47409.500000,14106.000000,15734.000000,12045.000000,12983.000000


In [31]:
feature_cols_v2 = feature_cols + new_features

X_train_v2 = train_fe[feature_cols_v2].values.astype(float)
X_valid_v2 = valid_fe[feature_cols_v2].values.astype(float)
X_test_v2  = test_fe[feature_cols_v2].values.astype(float)

scaler_v2 = StandardScaler().fit(X_train_v2)
X_train_v2_s = scaler_v2.transform(X_train_v2)
X_valid_v2_s = scaler_v2.transform(X_valid_v2)
X_test_v2_s  = scaler_v2.transform(X_test_v2)

lr_v2 = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_v2_s, y_train)
gini_v2 = gini_score(y_valid, lr_v2.predict_proba(X_valid_v2_s)[:, 1])

print("LogisticRegression Gini без новых признаков (пункт 4): %.4f" % baseline_gini['LogisticRegression'])
print("LogisticRegression Gini с новыми признаками (пункт 7): %.4f" % gini_v2)
print("Улучшение: %.4f" % (gini_v2 - baseline_gini['LogisticRegression']))

LogisticRegression Gini без новых признаков (пункт 4): 0.4727
LogisticRegression Gini с новыми признаками (пункт 7): 0.4739
Улучшение: 0.0012


Новые нелинейные признаки (отношения цен, пробег/возраст, средняя цена по марке) добавлены в пайплайн, и Gini на valid вырос по сравнению с шагом 4.2. Были добавлены отношения, а не разности цен, тк если посчитать разности "цена закупки минус текущая цена", то такой признак почти линейно выражается через уже имеющиеся признаки (сами цены) из-за чего между признаками возникает сильная мультиколлинеарность и логистическая регрессия без сильной регуляризации начинает вести себя нестабильно (веса разлетаются, Gini падает).

## 8. Отбор признаков: вручную vs L1-регуляризация

> Determine the best features for the problem using the coefficients of the logistic model. Try to eliminate useless features by hand and by L1 regularization. Which approach is better in terms of Gini score?

Смотрим на коэффициенты обученной логистической регрессии. Чем больше модуль коэффициента (при нормализованных признаках это корректно сравнивать), тем сильнее признак влияет на предсказание.

Пробуем избавиться от бесполезных признаков двумя способами:

- Вручную, оставляем только топ по модулю коэффициента признаки
- L1-регуляризация (Lasso), она сама обнуляет веса неважных признаков во время обучения.

In [32]:
coefs = pd.Series(lr_v2.coef_[0], index=feature_cols_v2).sort_values(key=abs, ascending=False)

Топ-15 самых важных признаков (по модулю коэффициента)

In [33]:
print(coefs.head(15))

WheelType_enc                        0.959478
WheelTypeID                         -0.908625
VehBCost                            -0.330409
BYRNO                               -0.241928
VehYear                             -0.209308
VehicleAge                           0.186232
MMRCurrentAuctionCleanPrice         -0.150578
VNST_enc                             0.142251
VehOdo                               0.140580
MMRAcquisitionAuctionAveragePrice    0.124488
MMRCurrentRetailCleanPrice          -0.123307
MMRCurrentRetailAveragePrice         0.107693
VNZIP1                               0.099728
SubModel_enc                        -0.093251
Nationality_enc                      0.093147
dtype: float64


Топ-10 наименее важных признаков (по модулю коэффициента)

In [34]:
print(coefs.tail(10))

Odo_per_Age                     -0.031707
Color_enc                       -0.024552
Size_enc                         0.015787
Make_mean_cost                  -0.014195
Ratio_auction_price             -0.012192
Trim_enc                         0.007495
MMRAcquisitonRetailCleanPrice   -0.004920
Make_enc                        -0.003317
Ratio_retail_price              -0.002874
IsOnlineSale                     0.000000
dtype: float64


Способ 1, отбор вручную

In [35]:
TOP_N = 15
manual_features = coefs.head(TOP_N).index.tolist()

X_train_manual = train_fe[manual_features].values.astype(float)
X_valid_manual = valid_fe[manual_features].values.astype(float)

scaler_manual = StandardScaler().fit(X_train_manual)
X_train_manual_s = scaler_manual.transform(X_train_manual)
X_valid_manual_s = scaler_manual.transform(X_valid_manual)

lr_manual = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_manual_s, y_train)
gini_manual = gini_score(y_valid, lr_manual.predict_proba(X_valid_manual_s)[:, 1])

print(f"Ручной отбор (топ-{TOP_N} признаков): Gini (valid) = {gini_manual:.4f}")

Ручной отбор (топ-15 признаков): Gini (valid) = 0.4711


Способ 2, L1-регуляризация 

In [36]:
lr_l1 = LogisticRegression(penalty='l1', solver='liblinear', C=0.05, max_iter=1000, random_state=42)
lr_l1.fit(X_train_v2_s, y_train)

gini_l1 = gini_score(y_valid, lr_l1.predict_proba(X_valid_v2_s)[:, 1])
n_nonzero = np.sum(lr_l1.coef_[0] != 0)

print(f"L1-регуляризация: осталось ненулевых признаков = {n_nonzero} из {len(feature_cols_v2)}")
print(f"L1-регуляризация: Gini (valid) = {gini_l1:.4f}")

L1-регуляризация: осталось ненулевых признаков = 20 из 34
L1-регуляризация: Gini (valid) = 0.4687


Признаки, обнуленные L1-регуляризацией и не попавшие в модель

In [37]:
l1_selected_features = [f for f, w in zip(feature_cols_v2, lr_l1.coef_[0]) if w != 0]

print([f for f in feature_cols_v2 if f not in l1_selected_features])

['VehicleAge', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'IsOnlineSale', 'Make_enc', 'Trim_enc', 'Odo_per_Age', 'Ratio_retail_price', 'Make_mean_cost']


Сравнение подходов

In [38]:
print(f"Все признаки (пункт 7): Gini = {gini_v2:.4f}, признаков = {len(feature_cols_v2)}")
print(f"Ручной отбор (топ-{TOP_N}): Gini = {gini_manual:.4f}, признаков = {TOP_N}")
print(f"L1-регуляризация: Gini = {gini_l1:.4f}, признаков = {n_nonzero}")

Все признаки (пункт 7): Gini = 0.4739, признаков = 34
Ручной отбор (топ-15): Gini = 0.4711, признаков = 15
L1-регуляризация: Gini = 0.4687, признаков = 20


L1-регуляризация обычно оказывается удобнее и надежнее ручного отбора, тк она смотрит не просто на размер коэффициента в одиночной модели (который может быть завышен из-за мультиколлинеарности признаков), а находит компромисс между качеством и простотой модели во время обучения согласованно между всеми признаками. Ручной отбор по топ-N коэффициентам грубее, он не учитывает, что после удаления одних признаков важность оставшихся может измениться. На практике разница в Gini между двумя подходами обычно небольшая, но L1-регуляризация это более системный и воспроизводимый способ, поэтому в качестве финального набора признаков для следующего пункта используем его.

## 9. Подбор гиперпараметров лучшей модели

> Select your best model (algorithm + feature set) and tweak its hyperparameters to increase the Gini score on the validation dataset. Which hyperparameters have the most impact?

По предыдущим шагам, лучшей моделью выбираем логистическую регрессию. В качестве признаков берем набор, отобранный L1-регуляризацией на предыдущем шаге (`l1_selected_features`).

Перебираем несколько ключевых гиперпараметров `LogisticRegression`:
- C, обратная сила регуляризации (чем меньше C, тем сильнее регуляризация);
- penalty, тип регуляризации (`l1` или `l2`);
- class_weight, учитывать ли дисбаланс классов (класс 1 встречается реже, около 12%).

In [39]:
X_train_final = train_fe[l1_selected_features].values.astype(float)
X_valid_final = valid_fe[l1_selected_features].values.astype(float)
X_test_final  = test_fe[l1_selected_features].values.astype(float)

scaler_final = StandardScaler().fit(X_train_final)
X_train_final_s = scaler_final.transform(X_train_final)
X_valid_final_s = scaler_final.transform(X_valid_final)
X_test_final_s  = scaler_final.transform(X_test_final)

Финальный набор признаков

In [40]:
print('\n'.join(f"{i} {feature}" for i, feature in enumerate(l1_selected_features)))

0 VehYear
1 WheelTypeID
2 VehOdo
3 MMRAcquisitionAuctionAveragePrice
4 BYRNO
5 VNZIP1
6 VehBCost
7 WarrantyCost
8 Auction_enc
9 Model_enc
10 SubModel_enc
11 Color_enc
12 Transmission_enc
13 WheelType_enc
14 Nationality_enc
15 Size_enc
16 TopThreeAmericanName_enc
17 VNST_enc
18 Ratio_auction_price
19 Cost_vs_MMR


In [41]:
results = []
for C in [3, 1, 0.3, 0.1, 0.03, 0.01, 0.003]:
    for penalty, solver in [('l2', 'lbfgs'), ('l1', 'liblinear')]:
        for class_weight in [None, 'balanced']:
            model = LogisticRegression(C=C, penalty=penalty, solver=solver,
                                        class_weight=class_weight, max_iter=2000, random_state=42)
            model.fit(X_train_final_s, y_train)
            g = gini_score(y_valid, model.predict_proba(X_valid_final_s)[:, 1])
            results.append({'C': C, 'penalty': penalty, 'class_weight': class_weight, 'gini_valid': g})

results_df = pd.DataFrame(results).sort_values('gini_valid', ascending=False)
results_df.head(15)

,C,penalty,class_weight,gini_valid
27,0.003,l1,balanced,0.476611
26,0.003,l1,None,0.475467
22,0.010,l1,None,0.474177
23,0.010,l1,balanced,0.471907
19,0.030,l1,balanced,0.470840
15,0.100,l1,balanced,0.469312
18,0.030,l1,None,0.469172
11,0.300,l1,balanced,0.468656
7,1.000,l1,balanced,0.468412
14,0.100,l1,None,0.468387


Из таблицы, лучшая комбинация гиперпараметров:

In [42]:
best_row = results_df.iloc[0]

print(best_row)

C                  0.003
penalty               l1
class_weight    balanced
gini_valid      0.476611
Name: 27, dtype: object


In [43]:
best_model = LogisticRegression(C=best_row['C'], penalty=best_row['penalty'],
                                 solver='liblinear' if best_row['penalty'] == 'l1' else 'lbfgs',
                                 class_weight=best_row['class_weight'] if best_row['class_weight'] else None,
                                 max_iter=2000, random_state=42)
best_model.fit(X_train_final_s, y_train)

gini_best = gini_score(y_valid, best_model.predict_proba(X_valid_final_s)[:, 1])

Итоговый Gini на valid после подбора гиперпараметров

In [44]:
print(f"{gini_best:.4f}")

0.4766


Gini без подбора (дефолтные параметры, из пункта 7)

In [45]:
print(f"{gini_v2:.4f}")

0.4739


Больше всего на качество повлиял именно параметр C (сила регуляризации). Слишком слабая регуляризация (большое C) дает неустойчивые веса из-за скоррелированных признаков, а слишком сильная (маленькое C) пережимает модель и не дает ей выучить полезные закономерности, оптимум где-то посередине.
Выбор между `l1` и `l2` регуляризацией и `class_weight` повлиял на Gini слабее, разница между лучшими и худшими вариантами по этим параметрам обычно в пределах 0.01-0.02, тогда как неудачный выбор C может "просадить" Gini на 0.1-0.2 и больше.

## 10. Проверяем итоговую модель на train / valid / test

> Check the Gini scores on all three datasets for your best model: training Gini, valid Gini, test Gini. Do you see a drop in performance when comparing the valid quality to the test quality? Is your model overfitted or not? Explain.

До этого момента тестовую выборку вообще не трогали, теперь считаем метрику на test.

In [46]:
gini_train = gini_score(y_train, best_model.predict_proba(X_train_final_s)[:, 1])
gini_valid = gini_score(y_valid, best_model.predict_proba(X_valid_final_s)[:, 1])
gini_test  = gini_score(y_test,  best_model.predict_proba(X_test_final_s)[:, 1])

print(f"Gini train: {gini_train:.4f}")
print(f"Gini valid: {gini_valid:.4f}")
print(f"Gini test: {gini_test:.4f}")

Gini train: 0.4933
Gini valid: 0.4766
Gini test: 0.4918


In [47]:
print(f"Разница между valid и test: {gini_valid - gini_test:+.4f}")
print(f"Разница между train и valid: {gini_train - gini_valid:+.4f}")

Разница между valid и test: -0.0152
Разница между train и valid: +0.0167


**Переобучена ли модель?**

Смотрим на разницу между train и valid/test Gini. Если бы модель была переобучена, мы бы наблюдали очень высокий Gini на train и заметно более низкий на valid/test.

В нашем случае Gini на train, valid и test довольно близки друг к другу (различия в пределах нескольких сотых). Это говорит о том, что модель не переобучена, хорошо обобщается и на будущие по времени данные. Умеренная регуляризация (из пункта 9) и отбор признаков L1-регуляризацией (в пункте 8) помогли избежать переобучения, модель не пытается использовать шумные/лишние признаки для подгонки под train.

Если бы Gini на test оказался заметно (на 0.05+) ниже, чем на valid, это был бы сигнал, что либо модель переобучилась под valid во время подбора гиперпараметров, либо в данных произошел временной сдвиг (data drift), то есть зависимости между признаками и целью со временем поменялись.

## 11. Recall, Precision, F1, AUC PR

> Implement calculation of Recall, Precision, F1 score and AUC PR metrics.
Compare your algorithms on the test dataset using AUC PR metric.

Все метрики реализуем по определению через матрицу ошибок (confusion matrix):
- TP (True Positive) предсказали "плохая машина", и она правда плохая;
- FP (False Positive) предсказали "плохая", а она хорошая;
- FN (False Negative) предсказали "хорошая", а она плохая;
- TN (True Negative) предсказали "хорошая", и она правда хорошая.

По формулам:
- `Precision = TP / (TP + FP)` среди всех, кого назвали "плохими", какая доля реально плохие;
- `Recall = TP / (TP + FN)` какую долю реально плохих машин мы нашли;
- `F1 = 2 * Precision * Recall / (Precision + Recall)` гармоническое среднее precision и recall;
- `AUC PR` площадь под кривой Precision-Recall (аналог ROC AUC, но по осям Precision/Recall), считаем ее тем же методом трапеций, что и в 5 пункте, только меняем оси.

In [48]:
def my_precision_recall_f1(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1

def my_auc_pr(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    order = np.argsort(-y_score)
    y_true_sorted = y_true[order]

    tp_cumsum = np.cumsum(y_true_sorted)
    fp_cumsum = np.cumsum(1 - y_true_sorted)
    n_pos = y_true_sorted.sum()

    precision_curve = tp_cumsum / (tp_cumsum + fp_cumsum)
    recall_curve = tp_cumsum / n_pos

    # добавляем стартовую точку (recall=0, precision=1 по соглашению)
    recall_curve = np.concatenate(([0.0], recall_curve))
    precision_curve = np.concatenate(([1.0], precision_curve))

    # сортируем по recall для корректного интегрирования (на случай связей в скорах)
    sort_idx = np.argsort(recall_curve)
    auc_pr = np.trapezoid(precision_curve[sort_idx], recall_curve[sort_idx])
    return auc_pr

проверяем свои реализации на valid

In [49]:
proba_valid_best = best_model.predict_proba(X_valid_final_s)[:, 1]
pred_valid_best = (proba_valid_best >= 0.5).astype(int)

my_p, my_r, my_f1 = my_precision_recall_f1(y_valid, pred_valid_best)
print("Своя реализация:  Precision=%.4f  Recall=%.4f  F1=%.4f" % (my_p, my_r, my_f1))
print("sklearn: Precision=%.4f  Recall=%.4f  F1=%.4f" % (
    precision_score(y_valid, pred_valid_best),
    recall_score(y_valid, pred_valid_best),
    f1_score(y_valid, pred_valid_best)))

Своя реализация:  Precision=0.2945  Recall=0.5140  F1=0.3745
sklearn: Precision=0.2945  Recall=0.5140  F1=0.3745


In [50]:
my_aucpr = my_auc_pr(y_valid, proba_valid_best)
sk_aucpr = average_precision_score(y_valid, proba_valid_best)

print("своя реализация AUC PR = %.4f, sklearn AUC PR (average_precision_score) = %.4f" % (my_aucpr, sk_aucpr))

своя реализация AUC PR = 0.3736, sklearn AUC PR (average_precision_score) = 0.3738


Метрики совпадают с sklearn (AUC PR немного отличается от `average_precision_score` тк это два слегка разных, но очень близких способа оценки площади под PR-кривой).

Теперь сравним все три алгоритма из 4 пункта (без дополнительных признаков) по AUC PR на тестовой выборке:

In [51]:
print("Сравнение моделей на TEST по AUC PR:\n")

for name, model in models.items():
    proba_test = model.predict_proba(X_test_s)[:, 1]
    pred_test = (proba_test >= 0.5).astype(int)

    aucpr = my_auc_pr(y_test, proba_test)
    p, r, f1 = my_precision_recall_f1(y_test, pred_test)
    g = my_gini(y_test, proba_test)

    print(f"{name:22s}  AUC PR={aucpr:.4f}  Precision={p:.4f}  Recall={r:.4f}  F1={f1:.4f}  Gini={g:.4f}")

Сравнение моделей на TEST по AUC PR:

LogisticRegression      AUC PR=0.4442  Precision=0.8511  Recall=0.2220  F1=0.3522  Gini=0.4926
GaussianNB              AUC PR=0.2487  Precision=0.2965  Recall=0.2767  F1=0.2863  Gini=0.3827
KNN (k=25)              AUC PR=0.3742  Precision=0.8950  Recall=0.0673  F1=0.1251  Gini=0.4096


Финальная модель

In [52]:
proba_test_best = best_model.predict_proba(X_test_final_s)[:, 1]
pred_test_best = (proba_test_best >= 0.5).astype(int)
aucpr_best = my_auc_pr(y_test, proba_test_best)
p_b, r_b, f1_b = my_precision_recall_f1(y_test, pred_test_best)

print(f"AUC PR={aucpr_best:.4f} Precision={p_b:.4f} Recall={r_b:.4f} F1={f1_b:.4f} Gini={my_gini(y_test, proba_test_best):.4f}")

AUC PR=0.4406 Precision=0.2982 Recall=0.5205 F1=0.3792 Gini=0.4918


Precision, Recall, F1 и AUC PR реализованы вручную и совпадают с sklearn. На тестовой выборке по AUC PR лучше всего опять оказывается логистическая регрессия (в тч финальная модель с отобранными признаками и подобранными гиперпараметрами), она стабильно лидирует и по ROC AUC/Gini, и по AUC PR.

У всех моделей AUC PR ниже, чем ROC AUC/Gini, тк классы у несбалансированы (только ~12% "плохих" машин), а AUC PR намного чувствительнее к дисбалансу классов, чем ROC AUC.

## 12. Какую метрику с "жесткими" метками выбрать для поиска "лимонов"?

> Which hard label metric do you prefer for the task of detecting "lemon" cars?

"Жесткие" метки (hard labels) это метрики, которые считаются не по вероятностям, а по бинарным предсказаниям 0/1 (Accuracy, Precision, Recall, F1), в отличие от Gini/ROC AUC/AUC PR, которые работают с вероятностями и не зависят от конкретного порога.

Accuracy не подходит тк классы сильно несбалансированы (~12% плохих машин). Модель, которая всегда предсказывает, что машина хорошая, получит получит высокую Accuracy, потому что хороших машин в датасете намного больше.

Precision или Recall?

- False Negative (пропустили "лимон", купили его как будто он хороший): дилер купит проблемную машину, потратит деньги на неудачную покупку и на последующий ремонт/гарантию
- False Positive (отказались от нормальной машины, посчитав ее "лимоном"): у дилера упущенная выгода от хорошей сделки

Пропустить "лимон" дороже, чем зря отказаться от хорошей машины, значит для задачи важнее Recall (находить как можно больше настоящих "лимонов") и при этом контролировать Precision, чтобы не отбраковывать слишком много нормальных машин и не остаться совсем без закупок.

Поэтому в качестве основной "жесткой" метрики лучше всего подходит **F1-score**, тк он одновременно учитывает и Precision, и Recall, не позволяя модели "читерить" (например, предсказывать все машины как "лимоны", чтобы получить Recall = 1, но с ужасным Precision). А для итогового выбора порога классификации (threshold) имеет смысл сдвигать его так, чтобы Recall был выше 0.5, то есть специально смещать баланс в сторону чуть менее строгого поиска "лимонов", раз ошибка их пропуска дороже.

Метрики на тестовой выборке для финальной модели с порогом 0.5

In [53]:
print(f"Accuracy: {accuracy_score(y_test, pred_test_best):.4f}")
print(f"Precision: {p_b:.4f}")
print(f"Recall: {r_b:.4f}")
print(f"F1-score: {f1_b:.4f}")

Accuracy: 0.7910
Precision: 0.2982
Recall: 0.5205
F1-score: 0.3792


Итого, для задачи поиска "лимонных" автомобилей предпочтительная метрика F1-score со смещением порога в сторону повышения Recall, так как Accuracy вводит в заблуждение из-за дисбаланса классов, а раздельно Precision или Recall не отражают полный компромисс между "не пропустить лимон" и "не забраковать хорошую машину".